# Battery Degradation Modeling with LSTM: Rollout Stability and Uncertainty Analysis

This notebook investigates long-horizon State of Health (SOH) prediction and Remaining Useful Life (RUL) estimation for lithium-ion batteries using LSTM-based sequence modeling.

**Key themes:**
- Long-horizon autoregressive prediction stability
- Baseline comparison to quantify LSTM's advantage at extended horizons
- Uncertainty estimation via Monte Carlo Dropout
- Multivariate feature modeling and its limitations

**Dataset:** NASA Li-ion Battery Aging Dataset (B0005)

In [ ]:
# ============================================================
# 1. Setup & Data Loading
# ============================================================

import os
import zipfile
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

from google.colab import drive
drive.mount('/content/drive')

# Unzip dataset
with zipfile.ZipFile('/content/drive/MyDrive/5.+Battery+Data+Set.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/battery_data')

with zipfile.ZipFile('/content/battery_data/5. Battery Data Set/1. BatteryAgingARC-FY08Q4.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/battery_data/batch1')

# Load B0005
mat = scipy.io.loadmat('/content/battery_data/batch1/B0005.mat')
battery = mat['B0005'][0][0]
cycles = battery['cycle'][0]
print(f'Total cycles: {len(cycles)}')

In [ ]:
# ============================================================
# 2. Extract Capacity & Compute SOH
# ============================================================

capacities = []
for cycle in cycles:
    if cycle['type'][0] == 'discharge':
        try:
            cap = cycle['data'][0][0]['Capacity'][0][0]
            capacities.append(cap)
        except:
            pass

capacities = np.array(capacities)
initial_capacity = capacities[0]
soh = capacities / initial_capacity

print(f'Discharge cycles: {len(capacities)}')
print(f'Initial capacity: {initial_capacity:.4f} Ah')
print(f'Final capacity:   {capacities[-1]:.4f} Ah')
print(f'Final SOH:        {soh[-1]:.4f}')

# Plot capacity degradation
plt.figure(figsize=(10, 4))
plt.plot(capacities)
plt.axhline(y=1.4, color='r', linestyle='--', label='EOL threshold (1.4 Ah)')
plt.xlabel('Discharge Cycle')
plt.ylabel('Capacity (Ah)')
plt.title('B0005 Battery Capacity Degradation')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3. Prepare Sliding Window Dataset
# ============================================================

window_size = 30

X, y = [], []
for i in range(len(soh) - window_size):
    X.append(soh[i:i+window_size])
    y.append(soh[i+window_size])

X = np.array(X)
y = np.array(y)

X_tensor = torch.FloatTensor(X).unsqueeze(-1)  # (N, 30, 1)
y_tensor = torch.FloatTensor(y)

split = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:split], X_tensor[split:]
y_train, y_test = y_tensor[:split], y_tensor[split:]

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')

In [ ]:
# ============================================================
# 4. LSTM Model Definition
# ============================================================

class BatteryLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out.squeeze()

In [ ]:
# ============================================================
# 5. Training
# ============================================================

model = BatteryLSTM()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=8, shuffle=True)

for epoch in range(500):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    if (epoch+1) % 100 == 0:
        model.eval()
        with torch.no_grad():
            test_loss = criterion(model(X_test), y_test)
        print(f'Epoch {epoch+1}/500 | Test Loss: {test_loss:.6f}')

In [ ]:
# ============================================================
# 6. Evaluation & SOH Prediction Visualization
# ============================================================

model.eval()
with torch.no_grad():
    all_pred = model(X_tensor).numpy()
    y_pred   = model(X_test).numpy()
    y_true   = y_test.numpy()

mae  = np.mean(np.abs(y_pred - y_true))
rmse = np.sqrt(np.mean((y_pred - y_true)**2))
print(f'Test MAE:  {mae:.4f}')
print(f'Test RMSE: {rmse:.4f}')

plt.figure(figsize=(12, 5))
plt.plot(range(len(soh)), soh, label='Ground Truth', alpha=0.8)
plt.plot(range(window_size, window_size+len(all_pred)), all_pred,
         label='LSTM Prediction', alpha=0.8)
plt.axvline(x=window_size+split, color='gray', linestyle='--', label='Train/Test Split')
plt.axhline(y=0.7, color='r', linestyle='--', label='EOL threshold (SOH=0.7)')
plt.xlabel('Cycle')
plt.ylabel('SOH')
plt.title('Battery SOH Prediction - B0005')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7. RUL Estimation
# ============================================================

eol_threshold = 0.7
actual_eol     = next(i for i, s in enumerate(soh) if s <= eol_threshold)
closest_idx    = np.argmin(np.abs(all_pred - eol_threshold))
predicted_eol  = closest_idx + window_size

print(f'Actual EOL cycle:     {actual_eol}')
print(f'Predicted EOL cycle:  {predicted_eol}')
print(f'RUL prediction error: {abs(actual_eol - predicted_eol)} cycles '
      f'({abs(actual_eol - predicted_eol)/actual_eol*100:.1f}%)')

In [ ]:
# ============================================================
# 8. Baseline Comparison
# ============================================================

X_train_np = X_train.squeeze(-1).numpy()
X_test_np  = X_test.squeeze(-1).numpy()

# Persistence: predict last observed value
persistence_pred = X_test_np[:, -1]

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_np, y_train.numpy())
lr_pred = lr.predict(X_test_np)

# MLP
mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp.fit(X_train_np, y_train.numpy())
mlp_pred = mlp.predict(X_test_np)

results = {
    'Persistence':        persistence_pred,
    'Linear Regression':  lr_pred,
    'MLP':                mlp_pred,
    'LSTM (ours)':        y_pred
}

print(f"{'Model':<22} {'MAE':>8} {'RMSE':>8}")
print('-' * 40)
for name, pred in results.items():
    m = np.mean(np.abs(pred - y_true))
    r = np.sqrt(np.mean((pred - y_true)**2))
    print(f"{name:<22} {m:>8.4f} {r:>8.4f}")

In [ ]:
# ============================================================
# 9. Rollout Horizon vs Error Analysis
# ============================================================
# Key insight: simple baselines perform well at short horizons,
# but LSTM's advantage becomes clear as prediction horizon grows.
# This mirrors compounding error dynamics studied in autoregressive
# sequence modeling (see: rnn-lstm-sequence-prediction).

def multi_step_rollout(model, initial_window, steps, use_lstm=True):
    window = initial_window.copy()
    predictions = []
    for _ in range(steps):
        if use_lstm:
            x = torch.FloatTensor(window[-window_size:]).unsqueeze(0).unsqueeze(-1)
            with torch.no_grad():
                pred = model(x).item()
        else:
            pred = window[-1]  # persistence
        predictions.append(pred)
        window = np.append(window, pred)
    return np.array(predictions)

start_idx      = split
initial_window = soh[start_idx - window_size:start_idx]
true_future    = soh[start_idx:]
max_horizon    = min(30, len(true_future))
horizons       = range(1, max_horizon + 1)

lstm_errors        = []
persistence_errors = []

for h in horizons:
    lstm_pred = multi_step_rollout(model, initial_window, h)
    pers_pred = multi_step_rollout(None, initial_window, h, use_lstm=False)
    true_vals = true_future[:h]
    lstm_errors.append(np.mean(np.abs(lstm_pred - true_vals)))
    persistence_errors.append(np.mean(np.abs(pers_pred - true_vals)))

plt.figure(figsize=(10, 4))
plt.plot(list(horizons), lstm_errors,        label='LSTM',        marker='o', markersize=3)
plt.plot(list(horizons), persistence_errors, label='Persistence', marker='x', markersize=3)
plt.xlabel('Prediction Horizon (cycles)')
plt.ylabel('MAE')
plt.title('Error Accumulation vs Prediction Horizon')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'Horizon  1 — LSTM: {lstm_errors[0]:.4f}  |  Persistence: {persistence_errors[0]:.4f}')
print(f'Horizon 10 — LSTM: {lstm_errors[9]:.4f}  |  Persistence: {persistence_errors[9]:.4f}')
print(f'Horizon 30 — LSTM: {lstm_errors[29]:.4f}  |  Persistence: {persistence_errors[29]:.4f}')

In [ ]:
# ============================================================
# 10. Multivariate Feature Modeling
# ============================================================
# Per-cycle statistical features: capacity, voltage, current, temperature.
# Finding: multivariate LSTM (MAE 0.0274) underperforms univariate (MAE 0.0079)
# due to limited training data (110 samples) and lossy statistical aggregation.
# This is a meaningful negative result — more features != better model.

def extract_discharge_features(cycles):
    features = []
    for cycle in cycles:
        if cycle['type'][0] == 'discharge':
            try:
                data    = cycle['data'][0][0]
                cap     = data['Capacity'][0][0]
                voltage = data['Voltage_measured'][0]
                current = data['Current_measured'][0]
                temp    = data['Temperature_measured'][0]
                features.append([
                    cap,
                    np.mean(voltage),
                    np.min(voltage),
                    np.mean(np.abs(current)),
                    np.mean(temp),
                    np.max(temp) - np.min(temp)
                ])
            except:
                pass
    return np.array(features)

features        = extract_discharge_features(cycles)
scaler          = MinMaxScaler()
features_scaled = scaler.fit_transform(features)
soh_mv          = features_scaled[:, 0]

X_mv, y_mv = [], []
for i in range(len(features_scaled) - window_size):
    X_mv.append(features_scaled[i:i+window_size])
    y_mv.append(soh_mv[i+window_size])

X_mv_tensor = torch.FloatTensor(np.array(X_mv))
y_mv_tensor = torch.FloatTensor(np.array(y_mv))
X_mv_train, X_mv_test = X_mv_tensor[:split], X_mv_tensor[split:]
y_mv_train, y_mv_test = y_mv_tensor[:split], y_mv_tensor[split:]

class BatteryLSTM_MV(nn.Module):
    def __init__(self, input_size=6, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

model_mv   = BatteryLSTM_MV()
opt_mv     = torch.optim.Adam(model_mv.parameters(), lr=0.0005)
loader_mv  = DataLoader(TensorDataset(X_mv_train, y_mv_train), batch_size=8, shuffle=True)

for epoch in range(500):
    model_mv.train()
    for xb, yb in loader_mv:
        opt_mv.zero_grad()
        criterion(model_mv(xb), yb).backward()
        opt_mv.step()

model_mv.eval()
with torch.no_grad():
    mv_pred = model_mv(X_mv_test).numpy()
    mv_true = y_mv_test.numpy()

mae_mv = np.mean(np.abs(mv_pred - mv_true))
print(f'Multivariate LSTM MAE: {mae_mv:.4f}')
print(f'Univariate LSTM MAE:   {mae:.4f}')
print('→ Negative result: statistical feature aggregation loses temporal detail.')

In [ ]:
# ============================================================
# 11. Uncertainty Estimation (Monte Carlo Dropout)
# ============================================================

class BatteryLSTM_MC(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, num_layers,
                               batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :])).squeeze()

model_mc  = BatteryLSTM_MC()
opt_mc    = torch.optim.Adam(model_mc.parameters(), lr=0.0005)
loader_mc = DataLoader(TensorDataset(X_train, y_train), batch_size=8, shuffle=True)

for epoch in range(500):
    model_mc.train()
    for xb, yb in loader_mc:
        opt_mc.zero_grad()
        criterion(model_mc(xb), yb).backward()
        opt_mc.step()
    if (epoch+1) % 100 == 0:
        print(f'Epoch {epoch+1}/500 done')

# MC Dropout inference: run 100 forward passes with dropout active
def mc_predict(model, X, n_samples=100):
    model.train()  # keep dropout active at inference
    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            preds.append(model(X).numpy())
    return np.array(preds)

mc_preds  = mc_predict(model_mc, X_tensor)
mean_pred = mc_preds.mean(axis=0)
std_pred  = mc_preds.std(axis=0)
cycles_pred = range(window_size, window_size + len(mean_pred))

# SOH prediction with uncertainty band
plt.figure(figsize=(12, 5))
plt.plot(range(len(soh)), soh, label='Ground Truth', alpha=0.8, color='steelblue')
plt.plot(cycles_pred, mean_pred, label='LSTM Mean Prediction', color='orange')
plt.fill_between(cycles_pred,
                 mean_pred - 2*std_pred,
                 mean_pred + 2*std_pred,
                 alpha=0.3, color='orange', label='95% Confidence Interval')
plt.axvline(x=window_size+split, color='gray', linestyle='--', label='Train/Test Split')
plt.axhline(y=0.7, color='r', linestyle='--', label='EOL threshold (SOH=0.7)')
plt.xlabel('Cycle')
plt.ylabel('SOH')
plt.title('Battery SOH Prediction with Uncertainty Estimation (MC Dropout)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Uncertainty over time
plt.figure(figsize=(12, 3))
plt.plot(cycles_pred, std_pred, color='purple')
plt.axvline(x=window_size+split, color='gray', linestyle='--', label='Train/Test Split')
plt.xlabel('Cycle')
plt.ylabel('Prediction Std')
plt.title('Prediction Uncertainty over Time')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'Mean uncertainty (train): {std_pred[:split].mean():.4f}')
print(f'Mean uncertainty (test):  {std_pred[split:].mean():.4f}')